In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
data_path = "/kaggle/input/datasets/thienmaivan/india-pines/Indian_pines_corrected.mat"
gt_path   = "/kaggle/input/datasets/thienmaivan/india-pines/Indian_pines_gt.mat"

In [ ]:
from scipy.io import loadmat
import numpy as np

data_mat = loadmat(data_path)
gt_mat = loadmat(gt_path)

print("Data keys:", data_mat.keys())
print("GT keys:", gt_mat.keys())

# PHAN 1: DATASET

1. Dataset provenance
2. Raw files
3. Cấu trúc Data
4. Dữ liệu thực tế
5. Các lớp mất cân bằng
6. Spatial analysis
7. Spectral analysis
8. Spectral-spatial patches
9. Preprocessing
10. Split strategy

# 1. Indian Pines Dataset

Indian Pines là một bộ dữ liệu ảnh siêu phổ được thu nhận bởi cảm biến AVIRIS
tại khu vực Indian Pines, Indiana, Hoa Kỳ.

Trong nghiên cứu này sử dụng hai file:

- `Indian_pines_corrected.mat`: chứa hyperspectral image cube đã được loại bỏ các spectral bands bị lỗi hoặc chịu ảnh hưởng mạnh bởi nhiễu/hấp thụ nước.
- `Indian_pines_gt.mat`: chứa ground-truth map, tức nhãn lớp tương ứng với từng vị trí pixel.

Dữ liệu corrected có kích thước không gian 145 × 145 pixel và gồm 200 spectral bands.

Ground truth gồm 16 lớp land-cover.
Label 0 biểu diễn các pixel không được gán nhãn và không được xem là một lớp phân loại.

2. Raw file

In [ ]:
print("Data keys:", data_mat.keys())
print("GT keys:", gt_mat.keys())

In [ ]:
print("=== DATA FILE METADATA ===")
print("Header :", data_mat["__header__"])
print("Version:", data_mat["__version__"])
print("Globals:", data_mat["__globals__"])

print("\n=== GT FILE METADATA ===")
print("Header :", gt_mat["__header__"])
print("Version:", gt_mat["__version__"])
print("Globals:", gt_mat["__globals__"])

In [ ]:
X = data_mat["indian_pines_corrected"]
y = gt_mat["indian_pines_gt"]

print("HSI variable:", type(X))
print("GT variable :", type(y))

3. Cấu trúc dữ liệu

# 3. Cấu trúc Data

Sau khi lấy hai biến chính từ file `.mat`:

- `X`: hyperspectral image cube
- `y`: ground-truth map

Ta kiểm tra shape, số chiều và kiểu dữ liệu của chúng để hiểu cấu trúc của bộ dữ liệu.

In [ ]:
print("=== HSI DATA ===")
print("Shape :", X.shape)
print("Ndim  :", X.ndim)
print("Dtype :", X.dtype)

print("\n=== GROUND TRUTH ===")
print("Shape :", y.shape)
print("Ndim  :", y.ndim)
print("Dtype :", y.dtype)

In [ ]:
pixel = X[50, 50, :]
print(pixel.shape)

In [ ]:
band = X[:, :, 30]
print(band.shape)

In [ ]:
assert X.shape[0] == y.shape[0]
assert X.shape[1] == y.shape[1]

print("Spatial dimensions match: PASSED")

4. Dữ liệu thực tế
- kiểm tra miền giá trị của dữ liệu HSI;
- kiểm tra dữ liệu có NaN hoặc Inf hay không;
- kiểm tra các label xuất hiện trong ground truth;
- xác định số pixel có nhãn và không có nhãn.

In [ ]:
print("=== HSI VALUE STATISTICS ===")

print("Minimum :", X.min())
print("Maximum :", X.max())
print("Mean    :", X.mean())
print("Std     :", X.std())

print("\nNaN count:", np.isnan(X).sum())
print("Inf count:", np.isinf(X).sum())

In [ ]:
coords = np.argwhere(y > 0)

row, col = coords[0]

spectrum = X[row, col, :]

print("Coordinate:", (row, col))
print("Label:", y[row, col])
print("Spectrum shape:", spectrum.shape)

print("\nFirst 20 spectral values:")
print(spectrum[:20])

In [ ]:
unique_labels = np.unique(y)

print("Unique labels:")
print(unique_labels)

print("Number of unique values:", len(unique_labels))

In [ ]:
total_pixels = y.size

labeled_pixels = np.sum(y > 0)
unlabeled_pixels = np.sum(y == 0)

print("Total pixels     :", total_pixels)
print("Labeled pixels   :", labeled_pixels)
print("Unlabeled pixels :", unlabeled_pixels)

print("\nLabeled ratio   :",
      labeled_pixels / total_pixels * 100)

print("Unlabeled ratio :",
      unlabeled_pixels / total_pixels * 100)

In [ ]:
assert np.isnan(X).sum() == 0
assert np.isinf(X).sum() == 0

assert y.min() == 0
assert y.max() == 16

assert labeled_pixels + unlabeled_pixels == total_pixels

print("Dataset value check: PASSED")

5. Mất cân bằng lớp
- xác định tên và ID của từng lớp;
- đếm số pixel của từng lớp;
- tính tỷ lệ phần trăm của mỗi lớp;
- xác định lớp lớn nhất và nhỏ nhất;
- định lượng mức mất cân bằng bằng imbalance ratio.

In [ ]:
class_names = [
    "Alfalfa",
    "Corn-notill",
    "Corn-mintill",
    "Corn",
    "Grass-pasture",
    "Grass-trees",
    "Grass-pasture-mowed",
    "Hay-windrowed",
    "Oats",
    "Soybean-notill",
    "Soybean-mintill",
    "Soybean-clean",
    "Wheat",
    "Woods",
    "Buildings-Grass-Trees-Drives",
    "Stone-Steel-Towers"
]

print("Number of classes:", len(class_names))

In [ ]:
class_counts = []

for cls in range(1, 17):
    count = np.sum(y == cls)
    class_counts.append(count)

for cls, name, count in zip(range(1, 17), class_names, class_counts):
    print(f"{cls:2d}. {name:30s}: {count}")

In [ ]:
import pandas as pd

total_labeled = np.sum(y > 0)

df_classes = pd.DataFrame({
    "Class ID": range(1, 17),
    "Class Name": class_names,
    "Samples": class_counts
})

df_classes["Percentage (%)"] = (
    df_classes["Samples"] / total_labeled * 100
)

df_classes

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    df_classes["Class Name"],
    df_classes["Samples"]
)

plt.xticks(rotation=90)
plt.ylabel("Number of samples")
plt.title("Class Distribution - Indian Pines")

plt.tight_layout()
plt.show()

In [ ]:
assert sum(class_counts) == np.sum(y > 0)
assert len(class_counts) == 16

print("Class distribution check: PASSED")

6. Spatial analysis
- trực quan hóa toàn bộ ground truth;
- xem từng lớp riêng biệt;
- xác định các lớp tập trung thành vùng hay phân tán;
- quan sát kích thước và hình dạng các vùng;
- chuẩn bị cho việc phân tích spatial split và nguy cơ leakage ở các bước sau.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# FALSE-COLOR COMPOSITE + GROUND TRUTH
# Indian Pines
# ============================================================

# Python index 0-based
bands = (29, 19, 9)

rgb = X[:, :, list(bands)].astype(np.float32)

# Contrast stretch từng band để ảnh dễ nhìn
for i in range(3):
    low, high = np.percentile(rgb[:, :, i], (2, 98))

    rgb[:, :, i] = np.clip(
        (rgb[:, :, i] - low) / (high - low + 1e-8),
        0,
        1
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# ------------------------------------------------------------
# HSI false-color
# ------------------------------------------------------------
axes[0].imshow(rgb)
axes[0].set_title(
    f"False-color composite (bands {bands})"
)
axes[0].axis("off")

# ------------------------------------------------------------
# Ground Truth
# ------------------------------------------------------------
im = axes[1].imshow(
    y,
    cmap="nipy_spectral",
    vmin=0,
    vmax=16
)

axes[1].set_title(
    "Ground truth (0=background, 1...16=lớp)"
)
axes[1].axis("off")

cbar = fig.colorbar(
    im,
    ax=axes[1],
    fraction=0.046,
    pad=0.04
)

cbar.set_ticks(range(0, 17))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

for cls in range(1, 17):
    mask = (y == cls)

    axes[cls - 1].imshow(mask, cmap="gray")
    axes[cls - 1].set_title(
        f"{cls}. {class_names[cls - 1]}\n"
        f"n = {np.sum(mask)}"
    )
    axes[cls - 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
spatial_stats = []

for cls in range(1, 17):
    coords = np.argwhere(y == cls)

    min_row, min_col = coords.min(axis=0)
    max_row, max_col = coords.max(axis=0)

    height = max_row - min_row + 1
    width = max_col - min_col + 1

    spatial_stats.append({
        "Class ID": cls,
        "Class Name": class_names[cls - 1],
        "Samples": len(coords),
        "Min Row": min_row,
        "Max Row": max_row,
        "Min Col": min_col,
        "Max Col": max_col,
        "BBox Height": height,
        "BBox Width": width
    })

df_spatial = pd.DataFrame(spatial_stats)
df_spatial

In [ ]:
df_spatial["BBox Area"] = (
    df_spatial["BBox Height"] *
    df_spatial["BBox Width"]
)

df_spatial["Spatial Density"] = (
    df_spatial["Samples"] /
    df_spatial["BBox Area"]
)

df_spatial[
    ["Class ID", "Class Name", "Samples", "BBox Area", "Spatial Density"]
]

In [ ]:
cls = 9  # Oats

coords = np.argwhere(y == cls)

print("Class:", class_names[cls - 1])
print("Number of pixels:", len(coords))

print("\nFirst 10 coordinates:")
print(coords[:10])


7. Spectral analysis
   
Mục tiêu của phần này là phân tích đặc trưng phổ của các pixel và các lớp trong Indian Pines.

- xem spectral signature của một pixel;
- so sánh phổ giữa các pixel cùng lớp;
- so sánh phổ giữa các lớp khác nhau;
- tính phổ trung bình của từng lớp;
- quan sát độ biến thiên phổ trong cùng một lớp;
- tìm các lớp có phổ tương đối giống nhau.

In [ ]:
coords = np.argwhere(y > 0)

row, col = coords[0]
label = y[row, col]

spectrum = X[row, col, :]

print("Coordinate:", (row, col))
print("Class:", label, "-", class_names[label - 1])
print("Spectrum shape:", spectrum.shape)

plt.figure(figsize=(10, 4))
plt.plot(range(1, 201), spectrum)

plt.xlabel("Spectral band")
plt.ylabel("Pixel value")
plt.title(
    f"Spectral Signature - "
    f"{label}. {class_names[label - 1]}"
)

plt.grid()
plt.show()

In [ ]:
cls = 2

coords = np.argwhere(y == cls)

r1, c1 = coords[0]
r2, c2 = coords[len(coords) // 2]

spectrum1 = X[r1, c1, :]
spectrum2 = X[r2, c2, :]

plt.figure(figsize=(10, 4))

plt.plot(
    range(1, 201),
    spectrum1,
    label=f"Pixel ({r1}, {c1})"
)

plt.plot(
    range(1, 201),
    spectrum2,
    label=f"Pixel ({r2}, {c2})"
)

plt.xlabel("Spectral band")
plt.ylabel("Pixel value")
plt.title(
    f"Two pixels from class {cls}: "
    f"{class_names[cls - 1]}"
)

plt.legend()
plt.grid()
plt.show()

In [ ]:
cls1 = 2
cls2 = 11

coord1 = np.argwhere(y == cls1)[0]
coord2 = np.argwhere(y == cls2)[0]

r1, c1 = coord1
r2, c2 = coord2

spectrum1 = X[r1, c1, :]
spectrum2 = X[r2, c2, :]

plt.figure(figsize=(10, 4))

plt.plot(
    range(1, 201),
    spectrum1,
    label=f"{cls1}. {class_names[cls1 - 1]}"
)

plt.plot(
    range(1, 201),
    spectrum2,
    label=f"{cls2}. {class_names[cls2 - 1]}"
)

plt.xlabel("Spectral band")
plt.ylabel("Pixel value")
plt.title("Spectral signatures from different classes")

plt.legend()
plt.grid()
plt.show()

In [ ]:
cls = 2

class_pixels = X[y == cls]

print("Class:", class_names[cls - 1])
print("Number of pixels:", class_pixels.shape[0])
print("Number of bands :", class_pixels.shape[1])

mean_spectrum = class_pixels.mean(axis=0)
std_spectrum = class_pixels.std(axis=0)

bands = np.arange(1, 201)

plt.figure(figsize=(10, 5))

plt.plot(
    bands,
    mean_spectrum,
    label="Mean spectrum"
)

plt.fill_between(
    bands,
    mean_spectrum - std_spectrum,
    mean_spectrum + std_spectrum,
    alpha=0.25,
    label="± 1 standard deviation"
)

plt.xlabel("Spectral band")
plt.ylabel("Pixel value")
plt.title(
    f"Mean Spectral Signature - "
    f"{cls}. {class_names[cls - 1]}"
)

plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

for cls in range(1, 17):
    class_pixels = X[y == cls]
    mean_spectrum = class_pixels.mean(axis=0)

    plt.plot(
        range(1, 201),
        mean_spectrum,
        label=f"{cls}. {class_names[cls - 1]}"
    )

plt.xlabel("Spectral band")
plt.ylabel("Mean pixel value")
plt.title("Mean Spectral Signatures of 16 Classes")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=8
)

plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
class_mean_spectra = []

for cls in range(1, 17):
    class_pixels = X[y == cls]
    mean_spectrum = class_pixels.mean(axis=0)

    class_mean_spectra.append(mean_spectrum)

class_mean_spectra = np.array(class_mean_spectra)

print("Shape:", class_mean_spectra.shape)


8. Spectral-spatial patches
Một pixel HSI chỉ chứa thông tin phổ tại một vị trí:

X[row, col, :] → vector 200 chiều.

Spectral-spatial patch mở rộng pixel trung tâm bằng cách lấy thêm các pixel lân cận.

Ví dụ với patch 5 × 5:

- kích thước không gian: 5 × 5 pixel;
- mỗi pixel có 200 spectral bands;
- patch có shape: 5 × 5 × 200.

Mục tiêu của phần này:

- hiểu cách trích xuất một patch;
- hiểu vai trò của pixel trung tâm;
- quan sát dữ liệu spectral-spatial thực tế;
- kiểm tra vấn đề tại biên ảnh;
- chuẩn bị cho preprocessing và 3D-HyperGAMO.

In [ ]:
patch_size = 5
half = patch_size // 2

print("Patch size:", patch_size)
print("Half size:", half)

In [ ]:
patch_size = 5
half = patch_size // 2

coords = np.argwhere(y > 0)

for row, col in coords:
    if (
        row >= half and
        row < X.shape[0] - half and
        col >= half and
        col < X.shape[1] - half
    ):
        break

patch = X[
    row-half : row+half+1,
    col-half : col+half+1,
    :
]

print("Center coordinate:", (row, col))
print("Center label:", y[row, col])
print("Class:", class_names[y[row, col] - 1])

print("\nPatch shape:", patch.shape)

In [ ]:
print("Spatial pixels:", patch.shape[0] * patch.shape[1])

print(
    "Total values:",
    patch.shape[0] *
    patch.shape[1] *
    patch.shape[2]
)

In [ ]:
center_in_patch = patch[half, half, :]

original_pixel = X[row, col, :]

print(
    "Center matches original:",
    np.array_equal(center_in_patch, original_pixel)
)

In [ ]:
label_patch = y[
    row-half : row+half+1,
    col-half : col+half+1
]

print("Center label:", y[row, col])

print("\nLabels inside patch:")
print(label_patch)

In [ ]:
band_idx = 50

plt.figure(figsize=(5, 5))

plt.imshow(
    patch[:, :, band_idx],
    cmap="gray"
)

plt.title(
    f"5×5 Spatial Patch - Band {band_idx + 1}"
)

plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, 201),
    patch[half, half, :],
    linewidth=2,
    label="Center pixel"
)

plt.plot(
    range(1, 201),
    patch[half, half + 1, :],
    alpha=0.7,
    label="Right neighbor"
)

plt.plot(
    range(1, 201),
    patch[half + 1, half, :],
    alpha=0.7,
    label="Bottom neighbor"
)

plt.xlabel("Spectral band")
plt.ylabel("Pixel value")
plt.title("Spectral signatures inside one spatial patch")

plt.legend()
plt.grid()
plt.show()

In [ ]:
patch_size = 5
half = patch_size // 2

X_padded = np.pad(
    X,
    (
        (half, half),
        (half, half),
        (0, 0)
    ),
    mode="constant"
)

print("Original shape:", X.shape)
print("Padded shape  :", X_padded.shape)

9. Preprocessing
    
Preprocessing biến dữ liệu HSI ban đầu thành dạng phù hợp hơn cho mô hình học máy.

Với Indian Pines và pipeline 3D-HyperGAMO, các bước cần quan tâm gồm:

1. chuyển dữ liệu sang dạng số thực;
2. chuẩn hóa giá trị spectral bands;
3. giảm chiều phổ nếu cần;
4. padding dữ liệu không gian;
5. trích xuất spectral-spatial patches.

Ở bước này chỉ phân tích preprocessing.
Việc chia train/validation/test được thực hiện ở phần tiếp theo.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# TIỀN XỬ LÝ KHÁM PHÁ — chưa dùng cho thực nghiệm chính thức

# Tạo bản sao dạng số thực, giữ nguyên X gốc
X_float = X.astype(np.float32)
H, W, B = X_float.shape

# Mỗi hàng là một pixel, mỗi cột là một dải phổ
X_2d = X_float.reshape(-1, B)

# Chuẩn hóa riêng từng dải phổ
scaler_demo = MinMaxScaler(feature_range=(-1, 1))
X_scaled_2d = scaler_demo.fit_transform(X_2d)

# Khôi phục hình dạng khối ảnh
X_scaled = X_scaled_2d.reshape(H, W, B)

print("Dữ liệu gốc:", X.shape)
print("Dạng bảng:", X_2d.shape)
print("Sau chuẩn hóa:", X_scaled.shape)
print("Min/max:", X_scaled.min(), X_scaled.max())


10. Split strategy

In [ ]:
from pathlib import Path
import numpy as np
from scipy.io import loadmat
import tensorflow as tf
import tf_keras as keras  # Keras 2, dùng cho mã gốc

def find_mat(filename):
    matches = [
        p for p in Path("/kaggle/input").rglob("*.mat")
        if p.name.lower() == filename.lower()
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Cần đúng một file {filename}, tìm thấy: {matches}"
        )
    return matches[0]

data_path = find_mat("Indian_pines_corrected.mat")
gt_path = find_mat("Indian_pines_gt.mat")

X = loadmat(data_path)["indian_pines_corrected"]
y = loadmat(gt_path)["indian_pines_gt"]

assert X.shape == (145, 145, 200)
assert y.shape == (145, 145)
assert np.isfinite(X).all()
assert np.array_equal(np.unique(y), np.arange(17))

gpus = tf.config.list_physical_devices("GPU")

print("Ảnh:", data_path)
print("Nhãn:", gt_path)
print("X:", X.shape, "| y:", y.shape)
print("Số pixel có nhãn:", int(np.count_nonzero(y)))
print("TensorFlow:", tf.__version__)
print("tf-keras:", keras.__version__)
print("GPU TensorFlow nhận được:", gpus)

if not gpus:
    raise RuntimeError("TensorFlow chưa nhận GPU; chưa chạy huấn luyện.")

print("OK — dữ liệu gốc và GPU đã sẵn sàng.")

clone repo

In [ ]:
%cd /kaggle/working

!git clone --recursive https://github.com/mhaut/3D-HyperGAMO.git

%cd /kaggle/working/3D-HyperGAMO/HSI-datasets

!python join_dsets.py

%cd /kaggle/working/3D-HyperGAMO

print("3D-HyperGAMO repo ready")

Kiểm tra dataset

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/3D-HyperGAMO/HSI-datasets/Classification")

files = [
    "indian_pines_corrected.mat",
    "indian_pines_corrected_gt.mat",
    "indianpines_disjoint_dset.mat",
]

for name in files:
    path = base / name
    print(name, "->", "OK" if path.exists() else "MISSING")

In [ ]:
# ============================================================
# KIỂM TRA PIPELINE INDIAN PINES THEO PAPER / REPO
# CHƯA TRAIN - CHƯA SỬA MODEL
# ============================================================

%cd /kaggle/working/3D-HyperGAMO

from pathlib import Path
import numpy as np
import scipy.io as sio
import auxil


# ------------------------------------------------------------
# 1. Preprocessing đúng logic repo:
#    200 bands -> PCA 40 -> MinMax [-1, 1]
# ------------------------------------------------------------

data, gt, num_class = auxil.loadData(
    "DIP",
    num_components=40
)

print("=== PREPROCESSING ===")
print("Cube sau PCA + MinMax :", data.shape)
print("GT                    :", gt.shape)
print("Số class              :", num_class)
print("Min/max               :", data.min(), data.max())


# ------------------------------------------------------------
# 2. Tạo toàn bộ patch 11x11x40
# ------------------------------------------------------------

patches, patch_labels = auxil.createImageCubes(
    data,
    gt,
    windowSize=11,
    removeZeroLabels=False
)

print("\n=== PATCHES ===")
print("All patches :", patches.shape)
print("Labels      :", patch_labels.shape)


# ------------------------------------------------------------
# 3. Load disjoint split của tác giả
# ------------------------------------------------------------

base = Path("HSI-datasets/Classification")

train_raw = sio.loadmat(
    base / "indianpines_disjoint_dset.mat"
)["indianpines_disjoint_dset"]

full_gt = sio.loadmat(
    base / "indian_pines_corrected_gt.mat"
)["indian_pines_gt"]


# Mapping đúng như code gốc
mapping = [
    0, 2, 3, 5, 6, 8, 10, 11, 12,
    14, 1, 4, 7, 9, 13, 15, 16
]

y_train_map = np.zeros_like(train_raw)

for old_label, new_label in enumerate(mapping):
    y_train_map[train_raw == old_label] = new_label


# Test = toàn bộ GT trừ các pixel đã dùng train
y_test_map = full_gt.copy()
y_test_map[y_train_map != 0] = 0


# ------------------------------------------------------------
# 4. Chuyển mask 2D -> vector để lấy đúng patch
# ------------------------------------------------------------

train_flat = y_train_map.reshape(-1)
test_flat = y_test_map.reshape(-1)

train_mask = train_flat != 0
test_mask = test_flat != 0

X_train = patches[train_mask]
X_test = patches[test_mask]

y_train = train_flat[train_mask] - 1
y_test = test_flat[test_mask] - 1


# Model Conv3D cần thêm channel cuối
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]


# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

assert not np.any(train_mask & test_mask)
assert X_train.shape[1:] == (11, 11, 40, 1)
assert X_test.shape[1:] == (11, 11, 40, 1)

assert y_train.min() == 0
assert y_train.max() == 15
assert y_test.min() == 0
assert y_test.max() == 15


print("\n=== DISJOINT SPLIT ===")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTrain/test overlap:", np.sum(train_mask & test_mask))


print("\n=== TRAIN SAMPLES / CLASS ===")

for cls in range(16):
    print(
        f"Class {cls + 1:2d}:",
        np.sum(y_train == cls)
    )


print("\n=== TEST SAMPLES / CLASS ===")

for cls in range(16):
    print(
        f"Class {cls + 1:2d}:",
        np.sum(y_test == cls)
    )


print("\nOK — pipeline 11×11×40 + disjoint split đã tạo được.")

In [ ]:
import numpy as np
import pandas as pd

train_mask = y_train_map != 0
gt_mask = full_gt != 0

# Train pixel nhưng standard GT lại là background
train_on_background = train_mask & (~gt_mask)

# Train pixel có trong GT và label khớp
train_match = (
    train_mask
    & gt_mask
    & (y_train_map == full_gt)
)

# Train pixel có trong GT nhưng label khác
train_mismatch = (
    train_mask
    & gt_mask
    & (y_train_map != full_gt)
)

print("=== KIỂM TRA DISJOINT MAP vs STANDARD GT ===")

print("Train samples:", np.sum(train_mask))
print("GT labeled pixels:", np.sum(gt_mask))

print(
    "Train nằm trên GT background:",
    np.sum(train_on_background)
)

print(
    "Train label khớp standard GT:",
    np.sum(train_match)
)

print(
    "Train label KHÔNG khớp standard GT:",
    np.sum(train_mismatch)
)


# Xem bảng đối chiếu label nếu có mismatch
if np.sum(train_mismatch) > 0:

    comparison = pd.crosstab(
        full_gt[train_mismatch],
        y_train_map[train_mismatch],
        rownames=["Standard GT"],
        colnames=["Disjoint train"]
    )

    print("\n=== LABEL MISMATCH TABLE ===")
    display(comparison)


# Kiểm tra số mẫu từng class trong disjoint train
print("\n=== DISJOINT TRAIN COUNTS ===")

for cls in range(1, 17):
    print(
        f"Class {cls:2d}:",
        np.sum(y_train_map == cls)
    )

In [ ]:
from pathlib import Path
import shutil

auxil_path = Path("/kaggle/working/3D-HyperGAMO/auxil.py")
backup_path = Path("/kaggle/working/3D-HyperGAMO/auxil_original.py")

# Backup code gốc một lần
if not backup_path.exists():
    shutil.copy2(auxil_path, backup_path)
    print("Đã backup auxil.py gốc.")

text = auxil_path.read_text()
lines = text.splitlines()

# Tìm block DIP
dip_start = next(
    i for i, line in enumerate(lines)
    if line.strip() == 'elif args.dataset == "DIP":'
)

# Tìm block DUP ngay sau DIP
dup_start = next(
    i for i in range(dip_start + 1, len(lines))
    if lines[i].strip() == 'elif args.dataset == "DUP":'
)

indent = lines[dip_start][:len(lines[dip_start]) - len(lines[dip_start].lstrip())]
body = indent + "    "

new_dip = f'''{indent}elif args.dataset == "DIP":

{body}data_path = os.path.join(
{body}    os.getcwd(),
{body}    'HSI-datasets/Classification/'
{body})

{body}y_train2 = sio.loadmat(
{body}    os.path.join(
{body}        data_path,
{body}        'indianpines_disjoint_dset.mat'
{body}    )
{body})['indianpines_disjoint_dset']

{body}y_test = sio.loadmat(
{body}    os.path.join(
{body}        data_path,
{body}        'indian_pines_corrected_gt.mat'
{body}    )
{body})['indian_pines_gt']

{body}y_train = copy.deepcopy(y_train2)

{body}mapping = [
{body}    0, 2, 3, 5, 6, 8, 10, 11, 12,
{body}    14, 1, 4, 7, 9, 13, 15, 16
{body}]

{body}for i, val in enumerate(mapping):
{body}    y_train[y_train2 == i] = val

{body}del y_train2

{body}# Disjoint test: loại toàn bộ vị trí đã dùng cho train
{body}y_test[y_train != 0] = 0

{body}# 2D masks -> vector
{body}y_train = y_train.reshape(-1)
{body}y_test = y_test.reshape(-1)

{body}# Lấy patch đúng theo train/test masks
{body}Xtrain = pixels[y_train != 0, :, :, :]
{body}Xtest = pixels[y_test != 0, :, :, :]

{body}# Loại background và chuyển label 1..16 -> 0..15
{body}y_train = y_train[y_train != 0] - 1
{body}y_test = y_test[y_test != 0] - 1

{body}# Shuffle train giống logic repo
{body}Xtrain, y_train = random_unison(
{body}    Xtrain,
{body}    y_train,
{body}    rstate=None
{body})

{body}# Conv3D cần channel cuối:
{body}# (N, S, S, B) -> (N, S, S, B, 1)
{body}Xtrain = Xtrain[..., np.newaxis]
{body}Xtest = Xtest[..., np.newaxis]
{body}pixels = pixels[..., np.newaxis]

{body}return (
{body}    Xtrain,
{body}    Xtest,
{body}    y_train,
{body}    y_test,
{body}    pixels,
{body}    labels,
{body}    num_bands,
{body}    num_class
{body})
'''

new_lines = (
    lines[:dip_start]
    + new_dip.splitlines()
    + lines[dup_start:]
)

new_text = "\n".join(new_lines) + "\n"

# Kiểm tra syntax trước khi ghi file
compile(new_text, str(auxil_path), "exec")

auxil_path.write_text(new_text)

print("Đã vá nhánh DIP.")
print("Syntax check: PASSED")

In [ ]:
import importlib
import auxil

importlib.reload(auxil)

class Args:
    dataset = "DIP"
    components = 40
    spatialsize = 11

args = Args()

result = auxil.loadDataFIX(args)

Xtr, Xte, ytr, yte, pixels_all, labels_all, bands, n_classes = result

print("=== LOAD DATA FIXED ===")
print("X_train :", Xtr.shape)
print("y_train :", ytr.shape)

print("X_test  :", Xte.shape)
print("y_test  :", yte.shape)

print("All patches:", pixels_all.shape)
print("Bands      :", bands)
print("Classes    :", n_classes)

print("Train labels:", ytr.min(), "->", ytr.max())
print("Test labels :", yte.min(), "->", yte.max())

assert Xtr.shape == (5689, 11, 11, 40, 1)
assert Xte.shape == (4711, 11, 11, 40, 1)
assert bands == 40
assert n_classes == 16

print("\nOK — DIP loader đã sẵn sàng.")

In [ ]:
# ============================================================
# PATCH COMPATIBILITY CHO KAGGLE HIỆN TẠI
# Không thay đổi thuật toán 3D-HyperGAMO
# ============================================================

from pathlib import Path
import shutil

repo = Path("/kaggle/working/3D-HyperGAMO")

files = {
    "main.py": repo / "main.py",
    "dense_net3D_SA.py": repo / "dense_net3D_SA.py",
    "dense_suppli.py": repo / "dense_suppli.py",
}

# Backup
for name, path in files.items():
    backup = path.with_suffix(path.suffix + ".original")

    if not backup.exists():
        shutil.copy2(path, backup)

print("Backup completed.")


# ------------------------------------------------------------
# main.py
# ------------------------------------------------------------

path = files["main.py"]
text = path.read_text()

text = text.replace(
    "from keras.optimizers import Adam",
    "from tf_keras.optimizers.legacy import Adam"
)

text = text.replace(
    "from keras.utils.np_utils import to_categorical",
    "from tf_keras.utils import to_categorical"
)

text = text.replace(
    "from keras.layers import Input",
    "from tf_keras.layers import Input"
)

text = text.replace(
    "from keras.models import Model",
    "from tf_keras.models import Model"
)

# NumPy 2.x không còn np.int
text = text.replace(
    "np.int(",
    "int("
)

compile(text, str(path), "exec")
path.write_text(text)

print("main.py: patched")


# ------------------------------------------------------------
# dense_suppli.py
# ------------------------------------------------------------

path = files["dense_suppli.py"]
text = path.read_text()

text = text.replace(
    "from keras.utils.np_utils import to_categorical",
    "from tf_keras.utils import to_categorical"
)

compile(text, str(path), "exec")
path.write_text(text)

print("dense_suppli.py: patched")


# ------------------------------------------------------------
# dense_net3D_SA.py
# ------------------------------------------------------------

path = files["dense_net3D_SA.py"]
text = path.read_text()

text = text.replace(
    "from keras import backend as K",
    "from tf_keras import backend as K"
)

text = text.replace(
    "from keras.layers import Input, Dropout, Dense, RepeatVector, Lambda, Reshape, Conv3D, Conv2D, Flatten, InputSpec",
    "from tf_keras.layers import Input, Dropout, Dense, RepeatVector, Lambda, Reshape, Conv3D, Conv2D, Flatten, InputSpec"
)

text = text.replace(
    "from keras.layers import BatchNormalization, Concatenate, Multiply, Add, Conv2DTranspose, GlobalAveragePooling2D",
    "from tf_keras.layers import BatchNormalization, Concatenate, Multiply, Add, Conv2DTranspose, GlobalAveragePooling2D, LeakyReLU, Softmax"
)

text = text.replace(
    "from keras.layers.advanced_activations import LeakyReLU, Softmax",
    ""
)

text = text.replace(
    "from keras.models import Model",
    "from tf_keras.models import Model"
)

compile(text, str(path), "exec")
path.write_text(text)

print("dense_net3D_SA.py: patched")

print("\nCompatibility patch: PASSED")

In [ ]:
%cd /kaggle/working/3D-HyperGAMO

!python main.py \
    --dataset DIP \
    --components 40 \
    --spatialsize 11 \
    --max_step 4

In [ ]:
%cd /kaggle/working/3D-HyperGAMO

!python main.py \
    --dataset DIP \
    --components 40 \
    --spatialsize 11 \
    --max_step 100

In [ ]:
from pathlib import Path
import shutil

repo = Path("/kaggle/working/3D-HyperGAMO")

src = repo / "main.py"
dst = repo / "main_monitor.py"

# Luôn tạo lại từ main.py hiện tại
shutil.copy2(src, dst)

text = dst.read_text()

# ============================================================
# 1. THÊM THƯ VIỆN CHO LOGGING + VISUALIZATION
# ============================================================

old_import = "import numpy as np"

new_import = """import numpy as np
import csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt"""

if old_import not in text:
    raise RuntimeError("Không tìm thấy import numpy.")

text = text.replace(old_import, new_import, 1)

lines = text.splitlines()


# ============================================================
# 2. TÌM VỊ TRÍ SAU KHI TẠO Pictures/
# ============================================================

pic_idx = next(
    i for i, line in enumerate(lines)
    if "os.makedirs(picPath)" in line
)

indent = lines[pic_idx][:len(lines[pic_idx]) - len(lines[pic_idx].lstrip())]

monitor_code = f'''
{indent}# ========================================================
{indent}# MONITORING — không tham gia cập nhật gradient
{indent}# ========================================================

{indent}monitorPath = os.path.join(
{indent}    savePath,
{indent}    "Monitor_" + args.dataset + "_" + str(args.max_step)
{indent})

{indent}if not os.path.exists(monitorPath):
{indent}    os.makedirs(monitorPath)

{indent}metricsCSV = os.path.join(monitorPath, "metrics.csv")

{indent}with open(metricsCSV, "w", newline="") as f:
{indent}    writer = csv.writer(f)
{indent}    writer.writerow([
{indent}        "step",
{indent}        "train_OA",
{indent}        "train_ACSA",
{indent}        "train_GM",
{indent}        "test_OA",
{indent}        "test_ACSA",
{indent}        "test_GM",
{indent}        "test_Kappa"
{indent}    ])

{indent}# RNG riêng chỉ dùng để tạo mẫu trực quan.
{indent}# Không làm thay đổi RNG của training.
{indent}vis_rng = np.random.RandomState(12345)


{indent}def _pseudo_rgb(patch):
{indent}    # 3 PCA components đầu -> pseudo RGB
{indent}    rgb = patch[:, :, :3].astype(np.float32).copy()

{indent}    for ch in range(3):
{indent}        mn = rgb[:, :, ch].min()
{indent}        mx = rgb[:, :, ch].max()

{indent}        if mx > mn:
{indent}            rgb[:, :, ch] = (
{indent}                rgb[:, :, ch] - mn
{indent}            ) / (mx - mn)
{indent}        else:
{indent}            rgb[:, :, ch] = 0.0

{indent}    return np.clip(rgb, 0, 1)


{indent}def save_monitor(step, saveStep, results):
{indent}    # ------------------------------------------------
{indent}    # CSV
{indent}    # ------------------------------------------------

{indent}    test_kappa = float(results[2]) / 100.0

{indent}    with open(metricsCSV, "a", newline="") as f:
{indent}        writer = csv.writer(f)

{indent}        writer.writerow([
{indent}            int(step),

{indent}            float(accSaveTr[saveStep]),
{indent}            float(acsaSaveTr[saveStep]),
{indent}            float(gmSaveTr[saveStep]),

{indent}            float(accSaveTs[saveStep]),
{indent}            float(acsaSaveTs[saveStep]),
{indent}            float(gmSaveTs[saveStep]),

{indent}            test_kappa
{indent}        ])


{indent}    # Các mốc đã có dữ liệu
{indent}    steps_hist = (
{indent}        np.arange(saveStep + 1) * resSamplePd
{indent}    )


{indent}    # ------------------------------------------------
{indent}    # ACSA / AA learning curve
{indent}    # ------------------------------------------------

{indent}    plt.figure(figsize=(8, 5))

{indent}    plt.plot(
{indent}        steps_hist,
{indent}        acsaSaveTr[:saveStep + 1],
{indent}        marker="o",
{indent}        label="Train ACSA"
{indent}    )

{indent}    plt.plot(
{indent}        steps_hist,
{indent}        acsaSaveTs[:saveStep + 1],
{indent}        marker="o",
{indent}        label="Test ACSA"
{indent}    )

{indent}    plt.xlabel("Step")
{indent}    plt.ylabel("ACSA")
{indent}    plt.ylim(0, 1.05)
{indent}    plt.grid(alpha=0.3)
{indent}    plt.legend()
{indent}    plt.tight_layout()

{indent}    plt.savefig(
{indent}        os.path.join(monitorPath, "ACSA_curve.png"),
{indent}        dpi=150
{indent}    )

{indent}    plt.close()


{indent}    # ------------------------------------------------
{indent}    # GM learning curve
{indent}    # ------------------------------------------------

{indent}    plt.figure(figsize=(8, 5))

{indent}    plt.plot(
{indent}        steps_hist,
{indent}        gmSaveTr[:saveStep + 1],
{indent}        marker="o",
{indent}        label="Train GM"
{indent}    )

{indent}    plt.plot(
{indent}        steps_hist,
{indent}        gmSaveTs[:saveStep + 1],
{indent}        marker="o",
{indent}        label="Test GM"
{indent}    )

{indent}    plt.xlabel("Step")
{indent}    plt.ylabel("Geometric Mean")
{indent}    plt.ylim(0, 1.05)
{indent}    plt.grid(alpha=0.3)
{indent}    plt.legend()
{indent}    plt.tight_layout()

{indent}    plt.savefig(
{indent}        os.path.join(monitorPath, "GM_curve.png"),
{indent}        dpi=150
{indent}    )

{indent}    plt.close()


{indent}    # ------------------------------------------------
{indent}    # OA learning curve
{indent}    # ------------------------------------------------

{indent}    plt.figure(figsize=(8, 5))

{indent}    plt.plot(
{indent}        steps_hist,
{indent}        accSaveTr[:saveStep + 1],
{indent}        marker="o",
{indent}        label="Train OA"
{indent}    )

{indent}    plt.plot(
{indent}        steps_hist,
{indent}        accSaveTs[:saveStep + 1],
{indent}        marker="o",
{indent}        label="Test OA"
{indent}    )

{indent}    plt.xlabel("Step")
{indent}    plt.ylabel("Overall Accuracy")
{indent}    plt.ylim(0, 1.05)
{indent}    plt.grid(alpha=0.3)
{indent}    plt.legend()
{indent}    plt.tight_layout()

{indent}    plt.savefig(
{indent}        os.path.join(monitorPath, "OA_curve.png"),
{indent}        dpi=150
{indent}    )

{indent}    plt.close()


{indent}    # ------------------------------------------------
{indent}    # Recall / TPR của từng class tại step hiện tại
{indent}    # ------------------------------------------------

{indent}    plt.figure(figsize=(10, 5))

{indent}    plt.bar(
{indent}        np.arange(1, c + 1),
{indent}        tprSaveTs[saveStep]
{indent}    )

{indent}    plt.xlabel("Relabeled class")
{indent}    plt.ylabel("Test recall / TPR")
{indent}    plt.ylim(0, 1.05)
{indent}    plt.xticks(np.arange(1, c + 1))
{indent}    plt.tight_layout()

{indent}    plt.savefig(
{indent}        os.path.join(
{indent}            monitorPath,
{indent}            "TPR_test_step_" + str(step).zfill(4) + ".png"
{indent}        ),
{indent}        dpi=150
{indent}    )

{indent}    plt.close()


{indent}    # ------------------------------------------------
{indent}    # Confusion matrix
{indent}    # ------------------------------------------------

{indent}    plt.figure(figsize=(7, 6))

{indent}    plt.imshow(
{indent}        confMatSaveTs[saveStep],
{indent}        interpolation="nearest"
{indent}    )

{indent}    plt.colorbar()
{indent}    plt.xlabel("Predicted class")
{indent}    plt.ylabel("True class")
{indent}    plt.title("Test confusion matrix - step " + str(step))
{indent}    plt.tight_layout()

{indent}    plt.savefig(
{indent}        os.path.join(
{indent}            monitorPath,
{indent}            "confusion_step_" + str(step).zfill(4) + ".png"
{indent}        ),
{indent}        dpi=150
{indent}    )

{indent}    plt.close()


{indent}    # ------------------------------------------------
{indent}    # Trực quan synthetic HSI patch mỗi 1000 step
{indent}    # ------------------------------------------------

{indent}    if step % 1000 == 0 and imbClsNum > 0:

{indent}        vis_class = 0

{indent}        vis_label = np.zeros(
{indent}            (1, c),
{indent}            dtype=np.float32
{indent}        )

{indent}        vis_label[:, vis_class] = 1

{indent}        vis_noise = vis_rng.normal(
{indent}            0,
{indent}            1,
{indent}            (1, latDim)
{indent}        ).astype(np.float32)

{indent}        intermediate = gen.predict(
{indent}            [vis_noise, vis_label],
{indent}            verbose=0
{indent}        )

{indent}        fake_vis = gen_processed[vis_class].predict(
{indent}            intermediate,
{indent}            verbose=0
{indent}        )[0, :, :, :, 0]

{indent}        real_vis = trainS[
{indent}            classMap[vis_class][0]
{indent}        ][:, :, :, 0]

{indent}        real_rgb = _pseudo_rgb(real_vis)
{indent}        fake_rgb = _pseudo_rgb(fake_vis)

{indent}        center = args.spatialsize // 2

{indent}        fig, axes = plt.subplots(
{indent}            1,
{indent}            3,
{indent}            figsize=(13, 4)
{indent}        )

{indent}        axes[0].imshow(real_rgb)
{indent}        axes[0].set_title(
{indent}            "Real pseudo-RGB"
{indent}        )
{indent}        axes[0].axis("off")

{indent}        axes[1].imshow(fake_rgb)
{indent}        axes[1].set_title(
{indent}            "Generated pseudo-RGB"
{indent}        )
{indent}        axes[1].axis("off")

{indent}        axes[2].plot(
{indent}            np.arange(1, bands + 1),
{indent}            real_vis[center, center, :],
{indent}            label="Real"
{indent}        )

{indent}        axes[2].plot(
{indent}            np.arange(1, bands + 1),
{indent}            fake_vis[center, center, :],
{indent}            label="Generated"
{indent}        )

{indent}        axes[2].set_xlabel("PCA component")
{indent}        axes[2].set_ylabel("Value")
{indent}        axes[2].legend()

{indent}        original_class = int(
{indent}            newOrder[vis_class]
{indent}        ) + 1

{indent}        fig.suptitle(
{indent}            "Minority class "
{indent}            + str(original_class)
{indent}            + " - step "
{indent}            + str(step)
{indent}        )

{indent}        plt.tight_layout()

{indent}        plt.savefig(
{indent}            os.path.join(
{indent}                monitorPath,
{indent}                "synthetic_step_"
{indent}                + str(step).zfill(4)
{indent}                + ".png"
{indent}            ),
{indent}            dpi=150
{indent}        )

{indent}        plt.close()


{indent}    print(
{indent}        "[MONITOR] Saved step",
{indent}        step,
{indent}        "->",
{indent}        monitorPath
{indent}    )
'''

lines[pic_idx + 1:pic_idx + 1] = monitor_code.splitlines()


# ============================================================
# 3. GỌI MONITOR SAU MỖI LẦN EVALUATE
# ============================================================

results_idx = next(
    i for i, line in enumerate(lines)
    if line.strip() ==
    "results = auxil.reports(pLabel, labelTs)[2]"
)

result_indent = (
    lines[results_idx][
        :len(lines[results_idx])
        - len(lines[results_idx].lstrip())
    ]
)

lines.insert(
    results_idx + 1,
    result_indent +
    "save_monitor(step, saveStep, results)"
)


# ============================================================
# 4. GHI FILE + KIỂM TRA SYNTAX
# ============================================================

new_text = "\n".join(lines) + "\n"

compile(
    new_text,
    str(dst),
    "exec"
)

dst.write_text(new_text)

print("main_monitor.py created.")
print("Syntax check: PASSED")
print("Training algorithm: UNCHANGED")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/3D-HyperGAMO/main_monitor.py")

lines = path.read_text().splitlines()

# ============================================================
# 1. Tìm block MONITORING mà ta đã chèn
# ============================================================

marker_idx = next(
    i for i, line in enumerate(lines)
    if "MONITORING — không tham gia cập nhật gradient" in line
)

# Tìm dòng cuối của save_monitor()
msg_idx = next(
    i for i in range(marker_idx, len(lines))
    if '"[MONITOR] Saved step"' in lines[i]
)

end_idx = next(
    i for i in range(msg_idx, len(lines))
    if lines[i].strip() == ")"
)

# ============================================================
# 2. Dedent toàn bộ monitor block 1 cấp
#    -> đưa nó ra khỏi "if not os.path.exists(picPath)"
# ============================================================

for i in range(marker_idx, end_idx + 1):
    if lines[i].startswith("    "):
        lines[i] = lines[i][4:]

# ============================================================
# 3. Với run 502, lưu synthetic cả Step 500
# ============================================================

for i in range(marker_idx, end_idx + 1):
    lines[i] = lines[i].replace(
        "if step % 1000 == 0 and imbClsNum > 0:",
        "if step % 500 == 0 and imbClsNum > 0:"
    )

# ============================================================
# 4. Kiểm tra syntax trước khi lưu
# ============================================================

new_text = "\n".join(lines) + "\n"

compile(
    new_text,
    str(path),
    "exec"
)

path.write_text(new_text)

print("Đã đưa save_monitor ra khỏi if picPath.")
print("Synthetic visualization: mỗi 500 step.")
print("Syntax check: PASSED")

In [ ]:
%cd /kaggle/working/3D-HyperGAMO

!python main_monitor.py \
    --dataset DIP \
    --components 40 \
    --spatialsize 11 \
    --max_step 502

In [ ]:
from pathlib import Path
from IPython.display import Image, display

monitor_dir = Path("/kaggle/working/3D-HyperGAMO/SavedModel/Monitor_DIP_502")

for name in [
    "ACSA_curve.png",
    "GM_curve.png",
    "OA_curve.png",
    "TPR_test_step_0000.png",
    "TPR_test_step_0500.png",
    "confusion_step_0000.png",
    "confusion_step_0500.png",
    "synthetic_step_0000.png",
    "synthetic_step_0500.png",
]:
    path = monitor_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print("MISSING:", name)

In [ ]:
%cd /kaggle/working/3D-HyperGAMO

!python main_monitor.py \
    --dataset DIP \
    --components 40 \
    --spatialsize 11 \
    --max_step 8000 \
    2>&1 | tee /kaggle/working/hypergamo_DIP_8000.log

In [ ]:
!ps -ef | grep '[m]ain_monitor.py'

print("\n=== LOG CUỐI ===")
!tail -n 30 /kaggle/working/hypergamo_DIP_8000.log

In [ ]:
from pathlib import Path
import pandas as pd

p = Path(
    "/kaggle/working/3D-HyperGAMO/"
    "SavedModel/Monitor_DIP_8000/metrics.csv"
)

df = pd.read_csv(p)

display(df.tail())

print("Step cuối đã lưu:", df["step"].iloc[-1])
print("Số mốc đánh giá:", len(df))

In [ ]:
# Toàn bộ lịch sử metric
display(df)

print("\n=== BEST TEST OA ===")
display(df.loc[[df["test_OA"].idxmax()]])

print("\n=== BEST TEST ACSA / AA ===")
display(df.loc[[df["test_ACSA"].idxmax()]])

print("\n=== BEST TEST GM ===")
display(df.loc[[df["test_GM"].idxmax()]])

print("\n=== BEST KAPPA ===")
display(df.loc[[df["test_Kappa"].idxmax()]])

Mô hình học rất nhanh và gần như bão hòa trên tập train từ khoảng step 2000. Tuy nhiên, kết quả test không tăng đều theo số bước huấn luyện. Test OA tốt nhất tại step 2000, trong khi ACSA và GM tốt nhất tại step 5000. Điều này cho thấy hiệu quả trên các lớp không đồng đều và cần phân tích sâu hơn theo từng lớp.


Vì đề tài tập trung vào mất cân bằng lớp, chỉ nhìn OA/AA/GM tổng thể chưa đủ. Ta cần xem Recall/TPR của từng lớp tại step 5000 — mốc có ACSA và GM tốt nhất — để xác định lớp nào 3D-HyperGAMO xử lý tốt và lớp nào vẫn thất bại

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

log_path = Path("/kaggle/working/hypergamo_DIP_8000.log")
text = log_path.read_text()

# Tìm TPR ngay sau "Test: Step: 5000"
pattern = (
    r"Test:\s*Step:\s*5000.*?"
    r"TPR:\s*\[([^\]]+)\]"
)

match = re.search(pattern, text, re.S)

if match is None:
    raise ValueError("Không tìm thấy TPR tại Test Step 5000.")

tpr_reordered = np.fromstring(
    match.group(1).replace("\n", " "),
    sep=" "
)

# Thứ tự class mà repo đã sắp xếp lại
new_order = np.array([
    8, 6, 0, 15, 12, 3, 4, 7,
    14, 11, 5, 2, 9, 13, 1, 10
])

# Đưa TPR về thứ tự class gốc 1..16
tpr_original = np.zeros(16)

for new_idx, original_idx in enumerate(new_order):
    tpr_original[original_idx] = tpr_reordered[new_idx]

# Vẽ
plt.figure(figsize=(11, 5))

plt.bar(
    np.arange(1, 17),
    tpr_original
)

plt.xlabel("Indian Pines class")
plt.ylabel("Recall / TPR")
plt.title("Per-class Recall at Step 5000")
plt.xticks(np.arange(1, 17))
plt.ylim(0, 1.05)

plt.show()

# In số cụ thể
for cls, value in enumerate(tpr_original, start=1):
    print(f"Class {cls:2d}: {value:.4f}")

Tại step 5000, phần lớn các lớp đạt Recall khá cao, trong đó lớp 1, 7, 8 và 14 đạt 100%. Tuy nhiên, lớp 15 chỉ đạt 44%, thấp hơn rõ rệt so với các lớp còn lại. Điều này cho thấy 3D-HyperGAMO vẫn còn một failure case đáng chú ý dù các chỉ số tổng thể khá tốt.

Recall chỉ cho biết lớp 15 bị dự đoán sai nhiều, nhưng chưa biết các mẫu của lớp 15 bị nhầm sang lớp nào. Vì vậy bước tiếp theo là xem confusion matrix tại step 5000.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

cm_path = Path(
    "/kaggle/working/3D-HyperGAMO/"
    "SavedModel/Monitor_DIP_8000/"
    "confusion_step_5000.png"
)

if cm_path.exists():
    display(Image(filename=str(cm_path)))
else:
    print("Không tìm thấy confusion matrix step 5000.")

Confusion matrix cho thấy phần lớn dự đoán tập trung trên đường chéo, nhưng vẫn tồn tại một số nhầm lẫn giữa các lớp. Tuy nhiên, do ma trận đang sử dụng thứ tự lớp đã được sắp xếp lại và hiển thị theo số lượng mẫu tuyệt đối, cần chuyển về thứ tự lớp gốc và chuẩn hóa theo hàng để phân tích chính xác từng lớp.

Bước tiếp theo nên làm confusion matrix normalized (%) + tên 16 lớp gốc. Khi đó mình sẽ nhìn rất rõ:

Class 15


→ đúng bao nhiêu %


→ nhầm sang class 2 bao nhiêu %


→ nhầm sang class 11 bao nhiêu %


...

In [ ]:
from pathlib import Path
from IPython.display import Image, display

monitor = Path(
    "/kaggle/working/3D-HyperGAMO/"
    "SavedModel/Monitor_DIP_8000"
)

for step in [0, 2000, 5000, 7500]:

    path = monitor / f"synthetic_step_{step:04d}.png"

    print(f"\n===== STEP {step} =====")

    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Không tìm thấy:", path.name)